<a href="https://colab.research.google.com/github/Payal355-yadav/ALFIDO_Customer_Behavior_Analysis.ipynb/blob/main/Sales_Performance_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sales and Order Data Analysis

## 1. Environment Setup and Data Loading

First, we'll import the necessary libraries and load the sales and order dataset. The code will handle both direct CSV uploads and ZIP archives containing CSV files, and will try multiple encodings to ensure successful loading.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import zipfile
import io

plt.style.use('default')
sns.set_theme()

print("Libraries imported successfully.")

Libraries imported successfully.


In [13]:
print("Please upload your CSV file (or a zip containing it) now:")
uploaded = files.upload()

df = None # Initialize df to None

if not uploaded:
    print("No file uploaded. Please upload your CSV or ZIP file.")
else:
    uploaded_filename = list(uploaded.keys())[0]
    print(f"Uploaded file: {uploaded_filename}")

    raw_data_bytes = None

    if uploaded_filename.endswith('.zip'):
        print("Detected a ZIP file. Unzipping...")
        with zipfile.ZipFile(io.BytesIO(uploaded[uploaded_filename]), 'r') as zf:
            csv_files = [name for name in zf.namelist() if name.endswith('.csv')]
            if not csv_files:
                print("No CSV file found inside the ZIP archive.")
            else:
                csv_filename_in_zip = csv_files[0]
                with zf.open(csv_filename_in_zip) as csv_file:
                    raw_data_bytes = csv_file.read()
                print(f"Successfully unzipped '{csv_filename_in_zip}'.")

    elif uploaded_filename.endswith('.csv'):
        print("Detected a CSV file.")
        raw_data_bytes = uploaded[uploaded_filename]

    else:
        print("Unsupported file format. Please upload a .csv or .zip file.")

    if raw_data_bytes is not None:
        decoded_csv_content = None
        tried_encodings = ['utf-8', 'latin1', 'cp1252']
        for encoding in tried_encodings:
            try:
                decoded_csv_content = raw_data_bytes.decode(encoding)
                print(f"Successfully decoded using '{encoding}' encoding.")
                break
            except UnicodeDecodeError:
                print(f"Decoding with '{encoding}' failed. Trying next encoding...")
            except Exception as e:
                print(f"An unexpected error occurred during decoding with '{encoding}': {e}")
                break

        if decoded_csv_content is None:
            print(f"Failed to decode CSV content with any of the tried encodings: {tried_encodings}. Please ensure the file is a valid CSV and check its encoding.")
        else:
            data_stream = io.StringIO(decoded_csv_content)
            try:
                df = pd.read_csv(data_stream)
                print("CSV read successfully with pandas.")
                print("First 5 rows of the dataset:")
                display(df.head())
            except Exception as e:
                print(f"Error reading CSV with pandas after decoding: {e}")
                print("This might be due to an incorrect separator or other CSV formatting issues. You might need to specify the 'sep' parameter.")
    else:
        print("No raw data bytes to process.")

if df is None:
    print("DataFrame could not be loaded. Please ensure a valid CSV or ZIP file was uploaded.")

Please upload your CSV file (or a zip containing it) now:


Saving superstore_final_dataset (1).csv.zip to superstore_final_dataset (1).csv (2).zip
Uploaded file: superstore_final_dataset (1).csv (2).zip
Detected a ZIP file. Unzipping...
Successfully unzipped 'superstore_final_dataset (1).csv'.
Decoding with 'utf-8' failed. Trying next encoding...
Successfully decoded using 'latin1' encoding.
CSV read successfully with pandas.
First 5 rows of the dataset:


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales
0,1,CA-2017-152156,8/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,8/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/6/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O Donnel,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O Donnel,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold N Roll Cart System,22.3680


## 2. Data Cleaning and Preparation

Now, we'll clean and prepare the data for analysis. This involves:
- Inspecting data types and missing values.
- Handling missing values.
- Removing duplicate records.
- Converting date columns to datetime objects.

In [15]:
if df is not None:
    print("Dataset Information:")
    df.info()
    print("\nMissing values per column:")
    display(df.isnull().sum())
    print("\nNumber of duplicate rows:", df.duplicated().sum())

    # --- Data Cleaning --- #

    # 1. Handle Missing Values
    # For this generic case, we'll drop columns that are entirely empty,
    # and for other columns, we'll fill numerical NaNs with mean/median and categorical with mode.
    initial_cols = df.shape[1]
    df.dropna(axis=1, how='all', inplace=True)
    if df.shape[1] < initial_cols:
        print(f"Dropped {initial_cols - df.shape[1]} columns that were entirely empty.")

    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype in ['int64', 'float64']:
                df[col].fillna(df[col].mean(), inplace=True) # or median()
                print(f"Filled missing numerical values in '{col}' with mean.")
            elif df[col].dtype == 'object':
                mode_val = df[col].mode()[0]
                df[col].fillna(mode_val, inplace=True)
                print(f"Filled missing categorical values in '{col}' with mode ({mode_val}).")
            else:
                print(f"Column '{col}' has missing values but is of an unhandled type ({df[col].dtype}). Skipping fill.")

    # 2. Remove Duplicate Records
    df.drop_duplicates(inplace=True)
    print(f"\nDuplicate rows removed. New number of rows: {df.shape[0]}")

    # 3. Convert Date Formats
    # Explicitly convert 'Order_Date' and 'Ship_Date' with the correct format
    date_cols_to_convert = {'Order_Date': '%d/%m/%Y', 'Ship_Date': '%d/%m/%Y'}

    for col, date_format in date_cols_to_convert.items():
        if col in df.columns and df[col].dtype == 'object':
            try:
                df[col] = pd.to_datetime(df[col], format=date_format, errors='coerce')
                print(f"Converted column '{col}' to datetime format using '{date_format}'.")
            except Exception as e:
                print(f"Could not convert column '{col}' to datetime with format '{date_format}': {e}")
                print("It might not be a date column or has mixed formats. Skipping conversion for this column.")
        else:
            print(f"Column '{col}' not found or not of object type. Skipping date conversion.")

    print("\nData cleaning and preparation complete. Updated DataFrame info:")
    df.info()
else:
    print("DataFrame not available for cleaning.")

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row_ID         9800 non-null   int64  
 1   Order_ID       9800 non-null   object 
 2   Order_Date     9800 non-null   object 
 3   Ship_Date      9800 non-null   object 
 4   Ship_Mode      9800 non-null   object 
 5   Customer_ID    9800 non-null   object 
 6   Customer_Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal_Code    9800 non-null   float64
 12  Region         9800 non-null   object 
 13  Product_ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub_Category   9800 non-null   object 
 16  Product_Name   9800 non-null   object 
 17  Sales          9800 non-null   

,0
Row_ID,0
Order_ID,0
Order_Date,0
Ship_Date,0
Ship_Mode,0
Customer_ID,0
Customer_Name,0
Segment,0
Country,0
City,0



Number of duplicate rows: 0

Duplicate rows removed. New number of rows: 9800
Converted column 'Order_Date' to datetime format using '%d/%m/%Y'.
Converted column 'Ship_Date' to datetime format using '%d/%m/%Y'.

Data cleaning and preparation complete. Updated DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row_ID         9800 non-null   int64         
 1   Order_ID       9800 non-null   object        
 2   Order_Date     9800 non-null   datetime64[ns]
 3   Ship_Date      9800 non-null   datetime64[ns]
 4   Ship_Mode      9800 non-null   object        
 5   Customer_ID    9800 non-null   object        
 6   Customer_Name  9800 non-null   object        
 7   Segment        9800 non-null   object        
 8   Country        9800 non-null   object        
 9   City           9800 non-null   object        
 10 

## 3. Create KPIs

Now we will calculate the Key Performance Indicators (KPIs) based on the available data.

As noted by the user, the dataset does not contain a 'Profit' column, so **Profit Margin** cannot be calculated. We will focus on Revenue and Average Order Value (AOV).

Conversion Metrics would require additional data on customer interactions or website visits, which is not available in this dataset.

In [16]:
if df is not None:
    print("Calculating KPIs...")

    # 1. Total Revenue
    # Assuming 'Sales' column represents revenue
    total_revenue = df['Sales'].sum()
    print(f"Total Revenue: ${total_revenue:,.2f}")

    # 2. Number of Unique Orders
    # Assuming each Order_ID represents a unique order
    num_unique_orders = df['Order_ID'].nunique()
    print(f"Number of Unique Orders: {num_unique_orders:,}")

    # 3. Average Order Value (AOV)
    # AOV = Total Revenue / Number of Unique Orders
    if num_unique_orders > 0:
        average_order_value = total_revenue / num_unique_orders
        print(f"Average Order Value (AOV): ${average_order_value:,.2f}")
    else:
        print("Cannot calculate AOV as there are no unique orders.")

    # 4. Profit Margin (Not calculable without 'Profit' column)
    print("\nNote: Profit Margin cannot be calculated as a 'Profit' column is not available in the dataset.")

    # 5. Conversion Metrics (Not calculable with current data)
    print("Note: Conversion Metrics require additional data (e.g., website traffic, lead data) not present in this dataset.")

    print("\nKPI calculation complete.")
else:
    print("DataFrame not available for KPI calculation.")

Calculating KPIs...
Total Revenue: $2,261,536.78
Number of Unique Orders: 4,922
Average Order Value (AOV): $459.48

Note: Profit Margin cannot be calculated as a 'Profit' column is not available in the dataset.
Note: Conversion Metrics require additional data (e.g., website traffic, lead data) not present in this dataset.

KPI calculation complete.
